# XGBoost documentation

Selected notes from the [XGBoost tutorials](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/index.html).

---

## [Introduction to Boosted Trees](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html#introduction-to-boosted-trees)

- _bias-variance tradeoff: simple and predictive model_
- _set of classification and regression trees (CART)_
- _A CART is a bit different from decision trees, in which the leaf only contains decision values. In CART, a real score is associated with each of the leaves_

### [Tree Boosting](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html#tree-boosting)

The **objective function** consists of two parts: **training loss** and **regularization term**,

\begin{equation}
\mathrm{obj}^{(t)} = \sum_{i=1}^n l(y_i, \hat{y}_i^{(t)}) + \sum_{k=1}^t \omega(f_k)
\end{equation}

### [Additive Training](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html#additive-training)

Prediction value at step $t$:

\begin{equation}
\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i) = \sum_{k=1}^t f_k(x_i)
\end{equation}

In the general case, we take the Taylor expansion of the loss function up to the second order:

\begin{equation}
\mathrm{obj}^{(t)} = \sum_{i=1}^n \left[ l(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right ] + \omega(f_t) + \mathrm{constant}
\end{equation}

where,

$$ g_i = \partial_{\hat{y}_i^{(t-1)}}  l\left(y_i, \hat{y}_i^{(t-1)} \right) $$

$$ h_i = \partial_{\hat{y}_i^{(t-1)}}^2  l\left(y_i, \hat{y}_i^{(t-1)} \right) $$

After we remove all the constants, the specific objective at step $t$ becomes

\begin{equation}
\sum_{i=1}^n \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right ] + \omega(f_t)
\end{equation}

This becomes our optimization goal for the new tree. One important advantage of this definition is that the value of the objective function only depends on 
$g_i$ and $h_i$. This is how XGBoost supports custom loss functions.

### [Model Complexity](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html#model-complexity)

We need to define the complexity of the tree $\omega(f)$. In order to do so, let us first refine the definition of the tree f(x) as

\begin{equation}
f_t(x) = w_{q(x)}, ~~ w \in \R^T, ~~ q\colon \R^d \rightarrow \{1, 2, \dots T\}.
\end{equation}

Here $w$ is the vector of scores on leaves, $q$ is a function assigning each data point to the corresponding leaf, and $T$ is the number of leaves. In XGBoost, we define the complexity as

\begin{equation}
\omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2
\end{equation}

### [The Structure Score](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html#the-structure-score)

Here is the _magical part of the derivation_. After re-formulating the tree model, we can write the objective value with the $t$-th tree as:

\begin{equation}
\mathrm{obj}^{(t)} \approx \sum_{i=1}^n \left[ g_i w_{q(x_i)} + \frac{1}{2} h_i w_{q(x_i)}^2 \right ] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2
\end{equation}

\begin{equation}
= \sum_{j=1}^T \left[ \left( \sum_{i\in I_j} g_i \right) w_j + \frac{1}{2} \left( \sum_{i\in I_j} h_i + \lambda \right) w_j^2 \right ] + \gamma T
\end{equation}

\begin{equation}
= \sum_{j=1}^T \left[ G_j w_j + \frac{1}{2} \left( H_j + \lambda \right) w_j^2 \right ] + \gamma T
\end{equation}

where $I_j = \{ i| q(x_i) = j\}$ is the set of indices of data points assigned to the $j$-th leaf, $G_j = \sum_{i\in I_j} g_i$, and $H_j = \sum_{i\in I_j} h_i$.

In this equation, $w_j$ are independent with respect to each other, the form $\left[\dots\right]$ is quadratic and the best $w_j$ for a given structure $q(x)$ and the best objective reduction we can get is:

\begin{equation}
w_j^* = - \frac{G_j}{H_j + \lambda}
\end{equation}

\begin{equation}
\mathrm{obj}^* = - \frac{1}{2} \sum_{j=1}^T \frac{G_j^2}{H_j + \lambda} + \gamma T
\end{equation}

The last equation measures how good a tree structure $q(x)$ is. This score is like the impurity measure in a decision tree, except that it also takes the model complexity into account.

### [Learn the tree structure](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html#learn-the-tree-structure)

We will try to optimize one level of the tree at a time. Specifically we try to split a leaf into two leaves, and the score it gains is

\begin{equation}
Gain = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L + H_R + \lambda} \right] - \gamma
\end{equation}

This formula can be decomposed as 1) the score on the new left leaf 2) the score on the new right leaf 3) The score on the original leaf 4) regularization on the additional leaf. We can see an important fact here: if the gain is smaller than $\gamma$, we would do better not to add that branch. This is exactly the **pruning** techniques in tree based models

> **Note**: Limitation of additive tree learning
>
> For edge cases, training results in a degenerate model because we consider only one feature dimension at a time. See [Can Gradient Boosting Learn Simple Arithmetic?](http://mariofilho.com/can-gradient-boosting-learn-simple-arithmetic/) for an example.

---

## [Introduction to Model IO](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/saving_model.html#introduction-to-model-io)

There are cases where we need to save something more than just the model itself. For example, in distributed training, XGBoost performs checkpointing operation. Or for some reasons, your favorite distributed computing framework decide to copy the model from one worker to another and continue the training in there. In such cases, the serialisation output is required to contain enough information to continue previous training without user providing any parameters again. We consider such scenario as memory snapshot (or memory based serialisation method) and distinguish it with normal model IO operation. Currently, memory snapshot is used in the following places:

- Python package: when the `Booster` object is pickled with the built-in `pickle` module.

To enable JSON format support for model IO (saving only the trees and objective), provide a filename with `.json` or `.ubj` as file extension.

```python
bst.save_model('model_file_name.json')
```

### [A note on backward compatibility of models and memory snapshots](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/saving_model.html#a-note-on-backward-compatibility-of-models-and-memory-snapshots)

If you’d like to store or archive your model for long-term storage, use `save_model` (Python).

On the other hand, memory snapshot (serialisation) captures many stuff internal to XGBoost, and its format is not stable and is subject to frequent changes.

If a model is persisted with `pickle.dump` (Python) or `saveRDS` (R), then the model may not be accessible in later versions of XGBoost.

### [Saving and Loading the internal parameters configuration](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/saving_model.html#saving-and-loading-the-internal-parameters-configuration)

XGBoost’s `C API`, `Python API` and `R API` support saving and loading the internal configuration directly as a JSON string. In Python package:

```python
bst = xgboost.train(...)
config = bst.save_config()
print(config)
```

You can load it back to the model generated by same version of XGBoost by:

```python
bst.load_config(config)
```

### [Difference between saving model and dumping model](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/saving_model.html#difference-between-saving-model-and-dumping-model)

XGBoost has a function called `dump_model` in the Booster class, which lets you to export the model in a readable format like `text`, `json` or `dot` (graphviz). The primary use case for it is for model interpretation and visualization, and is not supposed to be loaded back to XGBoost.

$$\dots$$

---

-

-

-

-

-